# NETBCI2026：本地最小样本接入核查

## tl;dr

本 companion notebook 复核已经下载的 NEMAR `nm000305 / v1.0.0 / sub-1`；不下载新 EEG，不训练模型，不创建计算资源。**本地结果：** `sub-1`，4 个 session、24 个 run，共 717 个实际 trial；事件类别：rest, right_hand。 下方保留实际执行输出。

任务为**右手运动想象与休息**。EEG 的跨天变化本身不构成人类学习或技能保持证据。SHU Ma2022 仍为 `pending author access`；本 notebook 不请求 `nm000288`，也不尝试 ZIP 密码。

## Context & Methods

目的：确认一个受试者的四个 session 是否可被 MNE 读取，检查 session/run、试次数、事件映射、时间单位和通道信息，并暴露转换版本与行为资料的限制。完整核查逻辑保存在 `scripts/check_netbci_subset.py`；这里执行该本地脚本并展示 JSON 结果，不复写数据适配器。

### Key Assumptions

- 只核查本地 `sub-1`，结果不能外推到全部 19 名受试者。
- `events.tsv` 的 `trial_type/value` 与对应 sidecar 是实际事件证据；README 声明若冲突需保留差异。
- NEMAR 为转换后的 derivative；不得将它和原始 Dataverse 版本直接混合。
- 在线 run 命中率包含会话内解码器配置的影响，不能自动等同无辅助神经技能。

### 1. 环境与输入

运行时使用项目的 Python 3.11 CPU 环境及已安装的 MNE/pandas。notebook 工具依赖单独安装到 `/workspace/.bci-tools/notebook-deps`，没有改变 `pyproject.toml` 或 `uv.lock`。

重跑时，选用含项目依赖的 Python kernel，先确保下列本地目录存在。数据缺失时应报告访问缺口，而不是在 notebook 中自动下载。

In [1]:
from pathlib import Path
from tempfile import TemporaryDirectory
from zoneinfo import ZoneInfo
from datetime import datetime
import hashlib
import json
import platform
import subprocess
import sys

import mne
import numpy as np
import pandas as pd
from IPython.display import Markdown, display

pd.set_option("display.max_colwidth", None)

REPO_ROOT = Path.cwd().resolve()
if not (REPO_ROOT / "scripts/check_netbci_subset.py").is_file():
    REPO_ROOT = REPO_ROOT.parent
assert (REPO_ROOT / "scripts/check_netbci_subset.py").is_file(), "Open from the repository or notebooks directory."
DATA_ROOT = REPO_ROOT / "data/netbci2026/nm000305/v1.0.0"
SOURCE_ROOT = REPO_ROOT / "research_logs/netbci2026_sources"
MANIFEST = SOURCE_ROOT / "download_manifest.json"
SUBJECT = "sub-1"
assert (DATA_ROOT / SUBJECT).is_dir(), "Local sub-1 subset is missing; this notebook does not download data."
assert MANIFEST.is_file(), "The download receipt is required for source and checksum checks."

runtime = {"Python": platform.python_version(), "MNE": mne.__version__, "NumPy": np.__version__, "pandas": pd.__version__}
display(pd.DataFrame(runtime.items(), columns=["Runtime component", "Actual version"]))

,Runtime component,Actual version
0,Python,3.11.16
1,MNE,1.10.2
2,NumPy,2.2.6
3,pandas,3.0.6


## Data

### 2. 来源、许可和转换版本

- [NEMAR 官方页面](https://www.nemar.org/dataset/nm000305)
- [NEMAR 数据 API](https://data.nemar.org/nm000305/)、[v1.0.0 manifest](https://data.nemar.org/nm000305/v1.0.0/manifest.json)
- [固定版本 DOI](https://doi.org/10.82901/nemar.nm000305.v1.0.0)
- [原始 Dataverse 来源](https://doi.org/10.57745/RBJRC7)
- [数据描述论文](https://doi.org/10.1038/s41597-026-08237-5)
- [CC BY 4.0 许可条款](https://creativecommons.org/licenses/by/4.0/)

下表读取已保存的官方文件。`GeneratedBy` 与 sourcedata 记录的是不同转换/检索阶段的版本号，不能把两者合成单一版本。使用时保留作者、论文和数据 DOI 及转换说明；本 notebook 不重新分发 EEG。

In [2]:
dataset_description = json.loads((DATA_ROOT / "dataset_description.json").read_text())
source_provenance = json.loads((DATA_ROOT / "sourcedata/sourcedata_provenance.json").read_text())
receipt = json.loads((SOURCE_ROOT / "access_receipt.json").read_text())
source_version_rows = [
    ("NEMAR dataset", receipt["dataset_id"]),
    ("NEMAR version", receipt["version"]),
    ("Dataset type", dataset_description["DatasetType"]),
    ("BIDS version declared", dataset_description["BIDSVersion"]),
    ("License declared", dataset_description["License"]),
    ("GeneratedBy MOABB", dataset_description["GeneratedBy"][0]["Version"]),
    ("Sourcedata retrieval MOABB", source_provenance["moabb_version_used"]),
    ("Sourcedata retrieval date", source_provenance["retrieved"]),
]
display(pd.DataFrame(source_version_rows, columns=["Source field", "Saved official value"]))

,Source field,Saved official value
0,NEMAR dataset,nm000305
1,NEMAR version,v1.0.0
2,Dataset type,derivative
3,BIDS version declared,1.9.0
4,License declared,CC-BY-4.0
5,GeneratedBy MOABB,1.8.0dev0
6,Sourcedata retrieval MOABB,v1.7.1-11-g9c6b1cc
7,Sourcedata retrieval date,2026-10-05


### 3. 已保存的访问与下载回执

下表展示本轮下载阶段的真实 HTTP 检查，**不是重跑 notebook 时再次联网的测试**。入口网页和数据 API 可用性分别记录。仅下载 `sub-1` 的必要文件及小型元数据，未下载完整 archive。

In [3]:
checked_at = datetime.fromisoformat(receipt["checked_at_utc"]).astimezone(ZoneInfo("Asia/Shanghai"))
display(Markdown(f"访问回执时间：**{checked_at:%Y-%m-%d %H:%M:%S %Z}**；受试者：`{receipt['scope']['subject']}`。"))
display(pd.DataFrame(receipt["http_checks"].items(), columns=["Saved access/checksum check", "Observed result"]))
display(pd.DataFrame([
    ("EEG files downloaded", receipt["scope"]["eeg_files"]),
    ("All necessary files downloaded", receipt["scope"]["downloaded_files"]),
    ("Bytes downloaded", receipt["scope"]["bytes"]),
    ("Whole dataset downloaded", receipt["scope"]["whole_dataset_downloaded"]),
], columns=["Downloaded scope", "Recorded value"]))

访问回执时间：**2026-10-09 20:26:28 CST**；受试者：`sub-1`。

,Saved access/checksum check,Observed result
0,landing_page_status,403
1,dataset_api_status,200
2,metadata_status,200
3,manifest_status,200
4,downloaded_file_checksums_all_match,True


,Downloaded scope,Recorded value
0,EEG files downloaded,24
1,All necessary files downloaded,169
2,Bytes downloaded,193462298
3,Whole dataset downloaded,False


## Results

### 4. 重放本地核查脚本

核查脚本只读取本地文件。输出放到临时目录，再读入内存；不覆盖原有审计日志。保存的 notebook 包含此次真实执行结果。

In [4]:
audit_script = REPO_ROOT / "scripts/check_netbci_subset.py"
with TemporaryDirectory(prefix="netbci-local-notebook-") as temporary_dir:
    report_path = Path(temporary_dir) / "subject1_audit.json"
    command = [sys.executable, str(audit_script), "--data-root", str(DATA_ROOT),
               "--manifest", str(MANIFEST), "--output", str(report_path)]
    process = subprocess.run(command, cwd=REPO_ROOT, check=True, capture_output=True, text=True)
    report = json.loads(report_path.read_text())
summary = report["summary"]
assert summary["subject"] == SUBJECT, "Unexpected audit subject."
assert summary["all_checks_pass"], "Audit failure: inspect checks before using the subset."
script_sha256 = hashlib.sha256(audit_script.read_bytes()).hexdigest()
display(Markdown(f"本地核查通过：**{summary['run_count']} 个 EEG 文件**。核查脚本 SHA-256：`{script_sha256}`"))

本地核查通过：**24 个 EEG 文件**。核查脚本 SHA-256：`9ccdde5c6ae7259fb2011e404f95b11f5eb472e02db8e136466061b7839472b3`

### 5. 受试者与 session/run 结构

以下数字来自实际本地读取和事件表汇总。按 session 汇总可直接检查纵向样本是否完整；本轮无需额外图形。

In [5]:
overview_rows = [
    ("Subject", summary["subject"]),
    ("Session count", summary["session_count"]),
    ("Run count", summary["run_count"]),
    ("EEG channel counts", ", ".join(map(str, summary["eeg_channel_counts"]))),
    ("Sampling frequencies (Hz)", ", ".join(map(str, summary["sampling_frequencies_hz"]))),
    ("Total observed trials", summary["total_trials"]),
    ("All required local checks passed", summary["all_checks_pass"]),
]
display(pd.DataFrame(overview_rows, columns=["Read result", "Observed value"]))
session_rows = []
for session in summary["sessions"]:
    runs = session["runs"]
    session_rows.append({"Session": session["session"], "Runs": len(runs) if isinstance(runs, list) else runs,
                         "Observed trials": session["trial_count"], **session["class_counts"]})
display(pd.DataFrame(session_rows))
display(Markdown(
    f"**本地结果：** `{summary['subject']}`，{summary['session_count']} 个 session、"
    f"{summary['run_count']} 个 run，共 {summary['total_trials']} 个实际 trial；"
    f"事件类别：{', '.join(sorted(summary['class_counts']))}。"
))

,Read result,Observed value
0,Subject,sub-1
1,Session count,4
2,Run count,24
3,EEG channel counts,74
4,Sampling frequencies (Hz),250.0
5,Total observed trials,717
6,All required local checks passed,True


,Session,Runs,Observed trials,right_hand,rest
0,01,6,180,90,90
1,02,6,179,90,89
2,03,6,180,90,90
3,04,6,178,90,88


**本地结果：** `sub-1`，4 个 session、24 个 run，共 717 个实际 trial；事件类别：rest, right_hand。

### 6. 事件语义、编码和时间单位

事件标签和数值来自实际 `events.tsv`，不以 README 的数值映射替代。时间单位来自相应 `events.json`；脚本核查数值与采样率及记录边界的关系。

In [6]:
event_rows = [{"trial_type": label, "Actual event value": value,
               "Observed trials": summary["class_counts"].get(label)}
              for label, value in sorted(summary["event_value_mapping"].items())]
display(pd.DataFrame(event_rows))
event_sidecars = sorted((DATA_ROOT / SUBJECT).glob("ses-*/eeg/*_events.json"))
event_metadata = [json.loads(path.read_text()) for path in event_sidecars]
unit_rows = [
    ("onset", sorted({metadata["onset"]["Units"] for metadata in event_metadata})),
    ("duration", sorted({metadata["duration"]["Units"] for metadata in event_metadata})),
    ("sample", event_metadata[0]["sample"]["Description"]),
]
display(pd.DataFrame(unit_rows, columns=["Event field", "Official sidecar units / description"]))

,trial_type,Actual event value,Observed trials
0,rest,1,357
1,right_hand,2,360


,Event field,Official sidecar units / description
0,onset,[s]
1,duration,[s]
2,sample,The event onset time in number of sampling points.First sample is 0.


### 7. 结果校验与版本差异

原始 README 的事件编号及 BrainVision 格式声明与 NEMAR EDF 及实际事件表存在差异。采用实际核查结果；保存差异，不能忽略或静默纠正上游元数据。

In [7]:
runs = report["runs"]
def passed_runs(field):
    return f"{sum(bool(run[field]) for run in runs)} / {len(runs)} runs"
check_rows = [
    ("All downloaded files verified against pinned manifest", summary["files_verified"]),
    ("Whole EEG signals finite", passed_runs("whole_signal_finite")),
    ("EDF annotations match TSV labels/onsets/durations", passed_runs("edf_tsv_annotations_match")),
    ("MNE event samples and values match TSV", passed_runs("mne_events_match")),
    ("Every trial is within its recording", passed_runs("all_trials_inside_recording")),
    ("Channel order stable across runs", summary["channel_order_stable"]),
    ("Maximum |onset × Hz − sample| (sampling points)", f"{max(run['event_sample_alignment_max_error'] for run in runs):.3e}"),
    ("Run trial counts observed", sorted({run["trial_count"] for run in runs})),
    ("Trial durations observed (s)", sorted({duration for run in runs for duration in run["trial_durations_s"]})),
    ("Trial lengths observed (samples)", sorted({length for run in runs for length in run["trial_lengths_samples"]})),
    ("EDF physical units observed", sorted({unit for run in runs for unit in run["physical_units"]})),
    ("MNE signal unit", sorted({run["mne_signal_unit"] for run in runs})),
]
display(pd.DataFrame(check_rows, columns=["Local validation", "Observed result"]))
readme_lines = (DATA_ROOT / "README.md").read_text().splitlines()
readme_events = next(line.strip() for line in readme_lines if line.strip().startswith("Events:"))
readme_format = next(line.strip() for line in readme_lines if line.strip().startswith("File format:"))
observed_formats = sorted({Path(run["file"]).suffix for run in runs})
display(pd.DataFrame([
    ("Event values", readme_events, json.dumps(summary["event_value_mapping"], ensure_ascii=False)),
    ("Signal format", readme_format, ", ".join(observed_formats)),
], columns=["Field", "NEMAR README declaration", "Observed local files"]))

,Local validation,Observed result
0,All downloaded files verified against pinned manifest,169
1,Whole EEG signals finite,24 / 24 runs
2,EDF annotations match TSV labels/onsets/durations,24 / 24 runs
3,MNE event samples and values match TSV,24 / 24 runs
4,Every trial is within its recording,24 / 24 runs
5,Channel order stable across runs,True
6,Maximum |onset × Hz − sample| (sampling points),3.638e-12
7,Run trial counts observed,"[29, 30]"
8,Trial durations observed (s),[5.0]
9,Trial lengths observed (samples),[1250]


,Field,NEMAR README declaration,Observed local files
0,Event values,"Events: right_hand=1, rest=2","{""right_hand"": 2, ""rest"": 1}"
1,Signal format,File format: BrainVision (BIDS),.edf


### 8. 行为资料边界

NEMAR 根 `participants.tsv` 是转换后的简化人口学表，不含原始版本的四次 session 行为成绩、六个 run 命中率及 channel/frequency 选择记录。下面仅列出实际字段，不把它称作完整行为记录；该转换表也含汇总或占位人口学值，不能作为已验证的个体协变量。

原始 Dataverse 的小型 participants 表提供每 session 六个 run 的光标命中百分比。这是真实在线任务成绩，但尚未逐项验证原始 participant ID 与 NEMAR `sub-1`、session/run 的跨版本对应，也没有已核查的 trial 级 hit/miss、光标轨迹、独立无辅助或延迟 retention 记录。因此当前不执行跨版本 join，不拟合学习曲线。后续关联应另做版本和身份校验。

In [8]:
participant_table = pd.read_csv(DATA_ROOT / "participants.tsv", sep="\t", keep_default_na=False)
display(pd.DataFrame({"Actual NEMAR participant columns": participant_table.columns}))
display(Markdown("原始行为来源及待核查差异见仓库 `docs/dataset_feasibility.md`；本 notebook 未读取或合并原始 Dataverse EEG。"))

,Actual NEMAR participant columns
0,participant_id
1,age
2,sex
3,hand
4,weight
5,height
6,group
7,species
8,bci_experience


原始行为来源及待核查差异见仓库 `docs/dataset_feasibility.md`；本 notebook 未读取或合并原始 Dataverse EEG。

## Takeaways

- NEMAR 的单受试者跨 session EEG 已完成本地 MNE 读取与事件核查，可继续作为最小研究接入候选。
- 下一步先确认可比较 session/run 的通道、事件及质量规则，并独立核实原始行为记录与 NEMAR 的对应关系；不自动扩大下载或启动训练。
- 任何后续跨天 EEG 漂移、离线解码变化或 run 命中率轨迹，均需与解码器重校准、测量、疲劳及练习等因素区分，不能直接宣称人类学习或 retention。
- SHU Ma2022 的原始数据仍为 `pending author access`。